# T04 — Garak + Hugging Face

**Objetivo:** usar **Garak** (probe → modelo → detector), revisar el **`.hitlog`** y entregar `reporte_<NUMERO_ALUMNO>.md`.

Flujo: explorar probes → exportar prompts → escanear (5× por prompt, 2 temperaturas) → analizar log.

## Token Hugging Face (cada alumno el suyo)

1. Cuenta en [huggingface.co](https://huggingface.co) (gratis).
2. Token **Read** en [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. **No compartir** el token.
4. **Colab:** Secrets → `HF_TOKEN` = tu `hf_...`.
5. **Local:** `export HF_INFERENCE_TOKEN=hf_...` o te lo pide la celda de setup.

In [1]:
!pip install garak -q

In [ ]:
import json, os, subprocess, sys
from collections import defaultdict
from pathlib import Path

NUMERO_ALUMNO = 1
HF_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
REPETICIONES = 5
TEMP_BAJA = 0.2
TEMP_ALTA = 0.9

try:
    from google.colab import userdata
    os.environ["HF_INFERENCE_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    if not os.environ.get("HF_INFERENCE_TOKEN"):
        import getpass
        os.environ["HF_INFERENCE_TOKEN"] = getpass.getpass("Tu HF token (Read): ")

token = os.environ.get("HF_INFERENCE_TOKEN", "").strip()
if not token.startswith("hf_"):
    raise ValueError("Falta token HF (Colab Secret HF_TOKEN o HF_INFERENCE_TOKEN).")

TALLER_DIR = Path(".").resolve()
RESULTADOS = TALLER_DIR / "resultados_garak"
RESULTADOS.mkdir(exist_ok=True)
GARAK_RUNS = Path.home() / ".local" / "share" / "garak" / "garak_runs"


def prompt_de_fila(row: dict) -> str:
    try:
        return row["prompt"]["turns"][0]["content"]["text"]
    except (KeyError, TypeError, IndexError):
        return str(row.get("prompt", ""))


def buscar_hitlog(prefix: str) -> Path | None:
    candidatos = []
    for base in (GARAK_RUNS, TALLER_DIR):
        if base.exists():
            candidatos.extend(base.rglob(f"*{prefix}*.hitlog.jsonl"))
    return max(candidatos, key=lambda p: p.stat().st_mtime) if candidatos else None


def correr_garak(temperature: float, suffix: str, probes: str) -> str:
    prefix = f"alumno_{NUMERO_ALUMNO}_{suffix}"
    cfg = {
        "run": {"generations": REPETICIONES, "soft_probe_prompt_cap": 2},
        "system": {"lite": True, "narrow_output": True},
        "plugins": {"extended_detectors": False, "probe_spec": probes},
    }
    cfg_path = RESULTADOS / f"{prefix}.json"
    cfg_path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    cmd = [
        sys.executable, "-m", "garak",
        "--config", str(cfg_path),
        "--target_type", "huggingface.InferenceAPI",
        "--target_name", HF_MODEL,
        "--report_prefix", prefix,
        "--generator_options", json.dumps({"temperature": temperature}),
        "--narrow_output",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, env=os.environ.copy(), check=False)
    return prefix


def analizar_hitlog(path: Path, mis_prompts: list[dict]) -> list[dict]:
    textos = {p["prompt"] for p in mis_prompts}
    filas = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if "probe" not in row or "score" not in row:
            continue
        if prompt_de_fila(row) in textos:
            filas.append(row)
    return filas


def resumir_evals(evals: list[dict], label: str) -> None:
    print(f"\n=== {label} ({len(evals)} evaluaciones de tus prompts) ===")
    by_probe = defaultdict(list)
    for e in evals:
        by_probe[e["probe"]].append(e)
    for probe, items in sorted(by_probe.items()):
        scores = [i["score"] for i in items]
        detectores = sorted({i.get("detector", "?") for i in items})
        hits = sum(1 for s in scores if s and s >= 0.5)
        print(f"  {probe}: {len(items)} evals, detector={detectores}, scores>={0.5}: {hits}")

## 1 — Explorar Garak

Garak organiza el pentest en **probes** (prompts) y **detectors** (miden la respuesta). Mira metadatos de los probes del taller:

In [ ]:
PROBES_TALLER = [
    "goodside.WhoIsRiley",
    "realtoxicityprompts.RTPBlank",
    "misleading.FalseAssertion",
    "snowball.GraphConnectivity",
    "packagehallucination.Python",
]

for probe in PROBES_TALLER:
    print("\n" + "=" * 60)
    subprocess.run(
        [sys.executable, "-m", "garak", "--plugin_info", f"probes.{probe}"],
        env=os.environ.copy(),
    )

## 2 — Prompts desde probes (`export_prompts_garak.py`)

Los prompts del taller **salen de Garak**. Ejecuta el script y revisa tu asignación (2 prompts por alumno):

In [ ]:
subprocess.run([sys.executable, str(TALLER_DIR / "export_prompts_garak.py")], check=True)

PROMPTS = json.loads((TALLER_DIR / "prompts_taller.json").read_text(encoding="utf-8"))
i0 = (NUMERO_ALUMNO - 1) * 2
MIS_PROMPTS = PROMPTS[i0 : i0 + 2]
if len(MIS_PROMPTS) < 2:
    raise ValueError(f"No hay 2 prompts para alumno {NUMERO_ALUMNO}.")

PROBES_ALUMNO = ",".join(dict.fromkeys(p["probe"] for p in MIS_PROMPTS))

print(f"Alumno {NUMERO_ALUMNO} -> ids {[p['id'] for p in MIS_PROMPTS]}")
print(f"Probes a escanear: {PROBES_ALUMNO}\n")
for p in MIS_PROMPTS:
    prev = p["prompt"][:80] if p["prompt"] else "(vacío)"
    print(f"  [{p['id']}] {p['probe']} / {p['categoria']} / {p['owasp']}")
    print(f"       {prev!r}")

## 3 — Escanear con Garak

`generations: 5` = **5 inferencias por prompt**. Corremos con **T baja** y luego **T alta**.

Target: `SmolLM2-360M-Instruct` vía Hugging Face Inference API.

In [ ]:
PREFIX_BAJA = correr_garak(TEMP_BAJA, "t0.2", PROBES_ALUMNO)
HITLOG_BAJA = buscar_hitlog(PREFIX_BAJA)
print(f"\nHitlog T={TEMP_BAJA}: {HITLOG_BAJA}")

In [ ]:
PREFIX_ALTA = correr_garak(TEMP_ALTA, "t0.9", PROBES_ALUMNO)
HITLOG_ALTA = buscar_hitlog(PREFIX_ALTA)
print(f"\nHitlog T={TEMP_ALTA}: {HITLOG_ALTA}")

## 4 — Analizar el `.hitlog`

Cada línea con `probe`, `detector` y `score` es una evaluación. Filtramos las filas que corresponden a **tus 2 prompts**.

Si no aparece hitlog, revisa la ruta impresa tras el escaneo (`~/.local/share/garak/garak_runs/`) o consulta al docente.

In [ ]:
if not HITLOG_BAJA:
    raise FileNotFoundError(
        "No se encontró hitlog T=0.2. Revisa el escaneo o la carpeta garak_runs."
    )

evals_baja = analizar_hitlog(HITLOG_BAJA, MIS_PROMPTS)
evals_alta = analizar_hitlog(HITLOG_ALTA, MIS_PROMPTS) if HITLOG_ALTA and HITLOG_ALTA.exists() else []

resumir_evals(evals_baja, f"T={TEMP_BAJA}")
resumir_evals(evals_alta, f"T={TEMP_ALTA}")

if evals_baja:
    e = evals_baja[0]
    print("\n--- Ejemplo de fila hitlog ---")
    print(f"probe: {e['probe']}")
    print(f"detector: {e.get('detector')}")
    print(f"score: {e.get('score')}")
    print(f"prompt: {prompt_de_fila(e)!r}")
    print(f"output: {e.get('output', {}).get('text', '')[:200]}")

## 5 — Reporte

Incluye: probe, detector, peor `score`, ¿cambió entre T=0.2 y T=0.9?, veredicto (¿vulnerable?).

In [ ]:
VEREDICTO = ""  # vulnerable | no_vulnerable | inconcluso
TEMPERATURA = """
¿Hubo más hits o scores altos con T=0.9 que con T=0.2?
"""
ANALISIS = """
Por cada prompt asignado: probe, detector, peor respuesta del hitlog, ¿aceptable en producción?
"""

md = f"""# Reporte Garak — alumno {NUMERO_ALUMNO}

## Target
- `{HF_MODEL}` (HF Inference API vía Garak)
- Probes escaneados: `{PROBES_ALUMNO}`
- Prompts asignados: {[p['id'] for p in MIS_PROMPTS]}
- Repeticiones (generations): {REPETICIONES}
- Hitlogs: `{HITLOG_BAJA}`, `{HITLOG_ALTA}`

## Temperatura
{TEMPERATURA.strip() or '(completar)'}

## Análisis de casos (desde hitlog)
{ANALISIS.strip() or '(completar)'}

## Veredicto
{VEREDICTO or '(completar)'}
"""

out = RESULTADOS / f"reporte_{NUMERO_ALUMNO}.md"
out.write_text(md, encoding="utf-8")
print(md)
print(f"\n-> {out}")